### Botlikh morphological transducer: coverage of texts with and without clitics

This notebook applies the transducer to four Botlikh stories from the collection [буйхалъи ххабарде](https://sverhees.github.io/botlikh_khabar/index.html) (compiled by G. Moroz, C. Naccarato and S. Verhees). The stories are stored in one table, `evaluation/texts/botlikh_khabar.tsv`; the script `scripts/make_khabar_texts.py` shows how this table was made from the files of the site. Two versions of the transducer are compared: without clitics and with clitics after a word of any part of speech.

**How to run.** Open the notebook in Google Colab and choose *Runtime → Run all*. The notebook installs `lexd` and HFST, takes the transducer, compiles it from the sources and then does the evaluation. Nothing in the notebook is random: the same sources and the same input file always give the same output. Checksums of the sources, of the input file and of the results are printed, so two runs can be compared directly.

#### 1. Settings

* `GITHUB_URL`: the repository with the transducer and the evaluation data.
* `GITHUB_COMMIT`: an optional commit hash that pins the exact version of the transducer.
* `PROJECT_DIR`: the folder of the repository (with the `Makefile` of the transducer).
* `OUT_DIR`: the folder for all tables and figures.
* `VARIANTS`: the two versions of the transducer that are compared. In the first one no clitics are allowed; in the second one a chain of up to three clitics can follow a word of any part of speech.


In [ ]:
GITHUB_URL = 'https://github.com/mbogautdinov/botlikh_transducer'
GITHUB_COMMIT = ''

PROJECT_DIR = 'botlikh_transducer'
OUT_DIR = 'eval_output'

VARIANTS = {
    'no clitics': 'bot_noclitics_analyzer',
    'all clitics': 'bot_allclitics_analyzer',
}

ANALYZERS = sorted(set(VARIANTS.values()))


#### 2. Tools

The transducer is compiled with `lexd` and HFST. If they are not installed yet, the cell installs them: first from the standard Ubuntu packages and, if `lexd` is not found there, from the package repository of the Apertium project. The helper `sh` runs a shell command, prints the end of its output and stops the notebook if the command fails. The versions of the tools are printed at the end, so that they are saved together with the results.


In [ ]:
import os
import shutil
import subprocess


def sh(cmd, check=True, tail=3000):
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip():
        print(r.stdout[-tail:])
    if check and r.returncode != 0:
        raise RuntimeError('command failed: ' + cmd)


TOOLS = ('lexd', 'hfst-twolc', 'hfst-compose-intersect', 'hfst-lookup', 'make')

if any(shutil.which(t) is None for t in TOOLS):
    sh('apt-get update -qq && apt-get install -y -qq hfst make', check=False)
    if shutil.which('lexd') is None:
        sh('apt-get install -y -qq lexd', check=False)
    if shutil.which('lexd') is None or shutil.which('hfst-twolc') is None:
        sh('wget -q https://apertium.projectjj.com/apt/install-nightly.sh -O /tmp/install-nightly.sh'
           ' && bash /tmp/install-nightly.sh && apt-get install -y -qq hfst lexd')

missing = [t for t in TOOLS if shutil.which(t) is None]
assert not missing, 'not installed: ' + ', '.join(missing)

sh('lexd --version 2>&1 | head -1; hfst-lookup --version 2>&1 | head -1; make --version | head -1; python3 --version')


#### 3. The transducer

The repository is cloned from GitHub. If the folder `PROJECT_DIR` already exists (for example, a local clone), it is used as it is. The commit of the repository is printed, so that the results can be traced to an exact version of the transducer.


In [ ]:
if not os.path.isdir(PROJECT_DIR):
    assert GITHUB_URL, 'set GITHUB_URL in the settings'
    sh('git clone -q ' + GITHUB_URL + ' ' + PROJECT_DIR)
    if GITHUB_COMMIT:
        sh('git -C ' + PROJECT_DIR + ' checkout -q ' + GITHUB_COMMIT)

assert os.path.isfile(os.path.join(PROJECT_DIR, 'Makefile')), 'the folder with the transducer was not found'
os.makedirs(OUT_DIR, exist_ok=True)
print('project:', os.path.abspath(PROJECT_DIR))
sh('git -C ' + PROJECT_DIR + ' log -1 --format="commit %H (%ad)" --date=short', check=False)


The transducer is rebuilt from scratch from the `lexd` and `twol` sources. `make` builds the analyzers, among them the two versions that are compared here. No Python scripts are called at this step: the lexicons are a part of the repository.


In [ ]:
sh('make -C ' + PROJECT_DIR + ' clean')
sh('make -C ' + PROJECT_DIR, tail=600)
sh('ls -la ' + ' '.join(os.path.join(PROJECT_DIR, a + '.hfstol') for a in ANALYZERS))


The cell below prints one checksum for all sources of the transducer (the `lexd` and `twol` files and the `Makefile`) and one for the input data. If these two values are the same on two computers, all results below must be the same as well.


In [ ]:
DATA = os.path.join(PROJECT_DIR, 'evaluation', 'texts', 'botlikh_khabar.tsv')
import glob
import hashlib


def sha256(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()


src_files = sorted(
    glob.glob(os.path.join(PROJECT_DIR, '*', '*.lexd'))
    + glob.glob(os.path.join(PROJECT_DIR, '*', '*.twol'))
    + glob.glob(os.path.join(PROJECT_DIR, '*.twol'))
    + [os.path.join(PROJECT_DIR, 'Makefile')]
)
h = hashlib.sha256()
for p in src_files:
    h.update(os.path.relpath(p, PROJECT_DIR).encode('utf-8'))
    with open(p, 'rb') as f:
        h.update(f.read())

print('transducer sources:', len(src_files), 'files, sha256', h.hexdigest())
print('input data:        ', os.path.relpath(DATA, PROJECT_DIR), 'sha256', sha256(DATA))


The function `lookup` sends a list of word forms to `hfst-lookup` and returns a dictionary `{form: [analyses]}`. An empty list means that the form is not recognised. The analyses of a form are sorted, so the order does not depend on the version of HFST.

Two more helpers read an analysis: `proposed_pos` returns the parts of speech that the analyses of a form propose (the first tag in capitals, such as `N` or `ADV`), and `clitic_tags` returns the tags of the clitics (the tags after `=`). A clitic that is written as a separate word has an analysis without a stem (`=<quot>`); it is counted as `CLITIC`.


In [ ]:
import re


def lookup(analyzer, forms):
    forms = [f for f in dict.fromkeys(forms) if f and not any(ch.isspace() for ch in f)]
    path = os.path.join(PROJECT_DIR, analyzer + '.hfstol')
    r = subprocess.run(['hfst-lookup', '-q', path], input='\n'.join(forms) + '\n',
                       capture_output=True, text=True, check=True)
    res = {f: [] for f in forms}
    for line in r.stdout.splitlines():
        parts = line.split('\t')
        if len(parts) >= 2 and parts[0] in res and not parts[1].endswith('+?'):
            if parts[1] not in res[parts[0]]:
                res[parts[0]].append(parts[1])
    return {f: sorted(a) for f, a in res.items()}


POS_TAG = re.compile(r'<([A-Z]+)>')
CLITIC_TAG = re.compile(r'=<([^>]+)>')


def proposed_pos(analyses):
    found = set()
    for a in analyses:
        m = POS_TAG.search(a)
        if m:
            found.add(m.group(1))
        elif a.startswith('='):
            found.add('CLITIC')
    return sorted(found)


def clitic_tags(analyses):
    return sorted({t for a in analyses for t in CLITIC_TAG.findall(a)})


for name, analyzer in VARIANTS.items():
    print(name, lookup(analyzer, ['хъуча', 'хъучиди', 'хъучала', 'гьикьила']))


#### 4. Figures: common style

All figures use the same two colours: blue for the transducer without clitics and orange for the transducer with clitics after all words. The pair was checked for the common forms of colour blindness. Grey is used for context, such as the total number of words. Values are also printed in tables next to every figure, so no number has to be read from a colour alone.

`grouped_bars` draws one group of bars per row of a table and one bar per transducer; `finish` adds the title, the axis label and the legend and removes everything that is not data.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

COLORS = {'no clitics': '#2a78d6', 'all clitics': '#eb6834'}
GREY = '#c3c2b7'
INK = '#0b0b0b'
INK2 = '#52514e'
MUTED = '#898781'
GRID = '#e1e0d9'
SURFACE = '#fcfcfb'
BLUES = LinearSegmentedColormap.from_list(
    'blues', [SURFACE, '#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b'])

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.edgecolor': GREY, 'axes.labelcolor': INK2, 'text.color': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'xtick.labelcolor': INK2, 'ytick.labelcolor': INK2,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'figure.dpi': 110,
})


def finish(ax, title, label='', horizontal=False, legend=False):
    ax.set_title(title, pad=10)
    ax.set_axisbelow(True)
    if horizontal:
        ax.set_xlabel(label)
        ax.grid(axis='x', color=GRID, linewidth=0.8)
        ax.spines['bottom'].set_visible(False)
        ax.tick_params(axis='both', length=0)
    else:
        ax.set_ylabel(label)
        ax.grid(axis='y', color=GRID, linewidth=0.8)
        ax.spines['left'].set_visible(False)
        ax.tick_params(axis='both', length=0)
    if legend:
        ax.legend(loc='best', handlelength=1.2, handleheight=1.0, borderaxespad=0.2)


def grouped_bars(ax, table, horizontal=False, fmt='{:.0f}', labels=True, total=None):
    groups = list(table.index)
    series = list(table.columns)
    pos = np.arange(len(groups))
    width = 0.8 / len(series)
    if total is not None:
        if horizontal:
            ax.barh(pos, total.values, height=0.86, color=GRID, label='all words', zorder=1)
        else:
            ax.bar(pos, total.values, width=0.86, color=GRID, label='all words', zorder=1)
    for i, s in enumerate(series):
        offset = (i - (len(series) - 1) / 2) * width
        values = table[s].values
        if horizontal:
            bars = ax.barh(pos + offset, values, height=width, color=COLORS[s], edgecolor=SURFACE,
                           linewidth=1.2, label=s, zorder=2)
        else:
            bars = ax.bar(pos + offset, values, width=width, color=COLORS[s], edgecolor=SURFACE,
                          linewidth=1.2, label=s, zorder=2)
        if labels:
            ax.bar_label(bars, labels=[fmt.format(v) for v in values], padding=2, fontsize=8, color=INK2)
    if horizontal:
        ax.set_yticks(pos)
        ax.set_yticklabels(groups)
        ax.invert_yaxis()
        ax.margins(x=0.12)
    else:
        ax.set_xticks(pos)
        ax.set_xticklabels(groups)
        ax.margins(y=0.12)


def save_figure(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(OUT_DIR, name + '.' + ext), dpi=200, bbox_inches='tight')


#### 5. The texts and their preprocessing

A sentence is split into tokens at spaces and punctuation marks (`SPLIT`). The hyphen is not in this list: a hyphen inside a word is kept, because Botlikh has reduplicated and compound words that are written with a hyphen; hyphens at the edges of a token are removed. Tokens without letters are dropped. Then the first letter of every token is made lower-case. Nothing else is changed; in particular, the palochka stays as it is written in the texts (the Latin capital I). The result is the table `tokens` with one row per token and the word form in the column `form`.


In [ ]:
import pandas as pd

texts = pd.read_csv(DATA, sep='\t')
print(texts.groupby(['text_id', 'title', 'recorded_by'], sort=False).size().rename('sentences'))

SPLIT = re.compile(r'[\s.,;:!?…«»"“”„()\[\]{}<>/\\–—]+')
HAS_LETTER = re.compile(r'[^\W\d_]')


def tokenize(sentence):
    out = []
    for t in SPLIT.split(str(sentence)):
        t = t.strip('-')
        if t and HAS_LETTER.search(t):
            out.append(t[:1].lower() + t[1:])
    return out


rows = []
for r in texts.itertuples():
    for k, tok in enumerate(tokenize(r.botlikh), 1):
        rows.append({'text_id': r.text_id, 'sentence_nr': r.sentence_nr, 'token_nr': k, 'form': tok})
tokens = pd.DataFrame(rows)

print(len(tokens), 'tokens,', tokens['form'].nunique(), 'types')
print(tokenize('ГьекIуб кастум гьикьи бета руша этала, бехутIу хъада багьихаб букIа.'))


#### 6. Analyses

Every form is analysed with both versions of the transducer. For each version the table gets the number of analyses, the analyses themselves, the proposed parts of speech and the tags of the clitics.


In [ ]:
results = {a: lookup(a, tokens['form'].tolist()) for a in ANALYZERS}

for name, analyzer in VARIANTS.items():
    res = results[analyzer]
    tokens['n (' + name + ')'] = tokens['form'].map(lambda f: len(res.get(f, [])))
    tokens['analyses (' + name + ')'] = tokens['form'].map(lambda f: ' | '.join(res.get(f, [])))
    tokens['pos (' + name + ')'] = tokens['form'].map(lambda f: proposed_pos(res.get(f, [])))
    tokens['clitics (' + name + ')'] = tokens['form'].map(lambda f: clitic_tags(res.get(f, [])))

tokens[['text_id', 'sentence_nr', 'form', 'n (no clitics)', 'n (all clitics)', 'analyses (all clitics)']].head(12)


#### 7. Metrics

* **coverage**: the share of tokens (or types) with at least one analysis;
* **analyses per recognised token**: how many analyses a recognised token gets on average, a measure of ambiguity.

The texts have no glosses, so the correctness of the analyses cannot be measured here. The numbers are given for all texts together and for every text.


In [ ]:
def metrics(df, label, names=None):
    out = []
    types = df.drop_duplicates('form')
    for name in (names or VARIANTS):
        n, nt = df['n (' + name + ')'], types['n (' + name + ')']
        out.append({
            'data': label, 'transducer': name,
            'tokens': len(df), 'recognised tokens': int((n > 0).sum()), 'token coverage': round((n > 0).mean(), 4),
            'types': len(types), 'recognised types': int((nt > 0).sum()), 'type coverage': round((nt > 0).mean(), 4),
            'analyses': int(n.sum()),
            'analyses per recognised token': round(n[n > 0].mean(), 2) if (n > 0).any() else 0,
        })
    return out


m = metrics(tokens, 'all texts')
for text_id, part in tokens.groupby('text_id', sort=False):
    m += metrics(part, text_id)
metrics_table = pd.DataFrame(m)
metrics_table


#### 8. With and without clitics: overview

The overview figure has four panels:

* **coverage**: the share of tokens and of types that get at least one analysis;
* **number of analyses per token**: how many tokens get no analysis, one analysis, two analyses and so on (the mean for the recognised tokens is given in the legend);
* **proposed parts of speech**: for every part of speech, the number of tokens that have at least one analysis with this part of speech (a token with several analyses can be counted for several parts of speech);
* **coverage in the parts of the data**: the same coverage of tokens, separately for every text or for every part of speech of the corpus.


In [ ]:
def count_table(df, key_col):
    out = {}
    for name in VARIANTS:
        counts = {}
        for keys in df[key_col + ' (' + name + ')']:
            for k in keys:
                counts[k] = counts.get(k, 0) + 1
        out[name] = counts
    table = pd.DataFrame(out).fillna(0).astype(int).sort_index()
    return table.loc[table.sum(axis=1).sort_values(ascending=False, kind='stable').index]


def overview_figure(df, group_col, group_title, name):
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8), constrained_layout=True)
    types = df.drop_duplicates('form')

    coverage = pd.DataFrame(
        {v: [100 * (df['n (' + v + ')'] > 0).mean(), 100 * (types['n (' + v + ')'] > 0).mean()] for v in VARIANTS},
        index=['tokens', 'types'])
    grouped_bars(axes[0, 0], coverage, fmt='{:.1f}')
    axes[0, 0].set_ylim(0, 100)
    finish(axes[0, 0], 'Coverage', '% with at least one analysis', legend=True)

    bins = ['0', '1', '2', '3', '4', '5+']
    dist = pd.DataFrame(
        {v: df['n (' + v + ')'].clip(upper=5).value_counts().reindex(range(6), fill_value=0).values for v in VARIANTS},
        index=bins)
    grouped_bars(axes[0, 1], dist)
    finish(axes[0, 1], 'Number of analyses per token', 'tokens')
    means = {v: df.loc[df['n (' + v + ')'] > 0, 'n (' + v + ')'].mean() for v in VARIANTS}
    handles, _ = axes[0, 1].get_legend_handles_labels()
    axes[0, 1].legend(handles, [v + ': mean ' + format(means[v], '.2f') + ' per recognised token' for v in VARIANTS],
                      loc='upper right', handlelength=1.2)
    axes[0, 1].set_xlabel('analyses')

    pos = count_table(df, 'pos')
    grouped_bars(axes[1, 0], pos, horizontal=True)
    finish(axes[1, 0], 'Proposed parts of speech', 'tokens with at least one such analysis', horizontal=True)

    by_group = pd.DataFrame(
        {v: 100 * df.groupby(group_col, sort=False)['n (' + v + ')'].apply(lambda s: (s > 0).mean()) for v in VARIANTS})
    sizes = df.groupby(group_col, sort=False).size()
    by_group = by_group.loc[sizes.sort_values(ascending=False).index]
    by_group.index = [str(g) + ' (' + str(sizes[g]) + ')' for g in by_group.index]
    grouped_bars(axes[1, 1], by_group, horizontal=True, fmt='{:.0f}')
    axes[1, 1].set_xlim(0, 100)
    finish(axes[1, 1], group_title, '% of tokens with at least one analysis', horizontal=True)

    save_figure(fig, name)
    plt.show()
    return coverage.round(2), dist, pos, by_group.round(1)


In this notebook the fourth panel shows the coverage for every text (the number of tokens is given in brackets).


In [ ]:
coverage, distribution, proposed, coverage_by_text = overview_figure(
    tokens, 'text_id', 'Coverage by text', 'texts_fig_overview')


#### 9. What the clitics add

The figure has two panels. The first one shows which clitics the transducer finds: for every clitic tag, the number of tokens that have at least one analysis with this clitic (several clitics have the same form, so one token is often counted for several tags). The second panel splits the tokens into three groups: recognised by both versions, recognised only when clitics are allowed, and not recognised at all. The second group is the gain in coverage.


In [ ]:
def clitics_figure(df, name):
    a, b = 'n (no clitics)', 'n (all clitics)'
    clitics = count_table(df, 'clitics')['all clitics']
    clitics = clitics[clitics > 0].head(15)
    groups = pd.Series({
        'recognised by both': int((df[a] > 0).sum()),
        'only with clitics': int(((df[a] == 0) & (df[b] > 0)).sum()),
        'not recognised': int((df[b] == 0).sum()),
    })

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True, gridspec_kw={'width_ratios': [1.15, 1]})

    bars = axes[0].barh(range(len(clitics)), clitics.values, height=0.62, color=COLORS['all clitics'],
                        edgecolor=SURFACE, linewidth=1.2)
    axes[0].bar_label(bars, padding=2, fontsize=8, color=INK2)
    axes[0].set_yticks(range(len(clitics)))
    axes[0].set_yticklabels(clitics.index)
    axes[0].invert_yaxis()
    axes[0].margins(x=0.1)
    finish(axes[0], 'Clitics found by the transducer with clitics', 'tokens with at least one such analysis', horizontal=True)

    group_colors = [COLORS['no clitics'], COLORS['all clitics'], GREY]
    bars = axes[1].bar(range(3), groups.values, width=0.5, color=group_colors, edgecolor=SURFACE, linewidth=1.2)
    axes[1].bar_label(bars, labels=[str(v) + ' (' + format(100 * v / len(df), '.1f') + '%)' for v in groups.values],
                      padding=2, fontsize=9, color=INK2)
    axes[1].set_xticks(range(3))
    axes[1].set_xticklabels(groups.index)
    axes[1].margins(y=0.12)
    finish(axes[1], 'Recognised with and without clitics', 'tokens')

    save_figure(fig, name)
    plt.show()
    return clitics.rename('tokens').rename_axis('clitic').reset_index(), groups.rename('tokens').rename_axis('group').reset_index()


clitics, groups = clitics_figure(tokens, 'texts_fig_clitics')


The forms that are recognised only with clitics, with their analyses (the most frequent ones first). The texts have no glosses, so these analyses are not checked here; some of them can be false, because a form of a verb can look like a noun with a clitic.


In [ ]:
a, b = 'n (no clitics)', 'n (all clitics)'
gained = (
    tokens[(tokens[a] == 0) & (tokens[b] > 0)].groupby(['form', 'analyses (all clitics)']).size()
    .rename('frequency').reset_index().sort_values(['frequency', 'form'], ascending=[False, True])
)
unknown = (
    tokens[tokens[b] == 0].groupby('form').size().rename('frequency').reset_index()
    .sort_values(['frequency', 'form'], ascending=[False, True])
)
print(len(gained), 'types are recognised only with clitics;', len(unknown), 'types are not recognised at all')
gained.head(20)


The most frequent forms that are not recognised by either version. Verb forms are expected in this list, because verbs are not implemented in the transducer.


In [ ]:
unknown.head(30)


#### 10. Saving the results

The tables are saved as text files and together in one Excel file, the figures as PNG and PDF. A cell that begins with `=` (the analysis of an isolated clitic) is stored as text, so that Excel does not read it as a formula. The checksums of the two main tables are printed for comparison with another run.


In [ ]:
out = tokens.copy()
for name in VARIANTS:
    for col in ('pos', 'clitics'):
        out[col + ' (' + name + ')'] = out[col + ' (' + name + ')'].map(', '.join)

tsv = {
    'texts_tokens.tsv': out,
    'texts_metrics.tsv': metrics_table,
    'texts_proposed_pos.tsv': proposed.reset_index().rename(columns={'index': 'pos'}),
    'texts_recognised_only_with_clitics.tsv': gained,
    'texts_not_recognised.tsv': unknown,
    'texts_clitics.tsv': clitics,
}
for name, df in tsv.items():
    df.to_csv(os.path.join(OUT_DIR, name), sep='\t', index=False)
    print(sha256(os.path.join(OUT_DIR, name)), name)

with pd.ExcelWriter(os.path.join(OUT_DIR, 'texts_results.xlsx'), engine='openpyxl') as xw:
    metrics_table.to_excel(xw, sheet_name='metrics', index=False)
    out.to_excel(xw, sheet_name='tokens', index=False)
    gained.to_excel(xw, sheet_name='only with clitics', index=False)
    unknown.to_excel(xw, sheet_name='not recognised', index=False)
    for ws in xw.sheets.values():
        for row in ws.iter_rows(min_row=2):
            for c in row:
                if c.data_type == 'f':
                    c.data_type = 's'
print('written:', sorted(os.listdir(OUT_DIR)))


All tables and figures are in `OUT_DIR`. In Colab the folder is packed into one archive and downloaded.


In [ ]:
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
try:
    from google.colab import files
    files.download(OUT_DIR + '.zip')
except ImportError:
    print('results:', os.path.abspath(OUT_DIR))
